# Module 3: Intent Classifier

**Goal:** classify what the customer actually wants, to route the message to the correct
handling path (small talk / order status / order management / billing / account / complaint /
out of scope).

**Dataset:** [`bitext/Bitext-customer-support-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)
— 26,872 instruction/response pairs across 27 intents and 10 categories. The `intent` column is
already gold-labeled, so this is genuine **supervised classification**, not zero/few-shot.

**Approach:** traditional ML — word-level TF-IDF (unigrams + bigrams) + `LinearSVC` with
`class_weight='balanced'` to handle the natural class imbalance across 27 intents.

**Why word-level TF-IDF here (unlike Module 1's char n-grams)?** Intent is carried by content
words and short phrases ("cancel", "refund", "track my order"), not by sub-word letter
patterns — the opposite signal from language identification.

**Two-stage label design:** we train and evaluate on the **fine-grained gold labels** (27
classes) — this is what the dataset actually supervises and is what should be reported/
defended at assessment — then apply a **deterministic mapping** (`src/intent_mapping.py`) down
to 7 coarse routing buckets used by the orchestration pipeline. `complaint`/`review` always map
to the priority `complaint` bucket regardless of classifier confidence, per the brief.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src import intent_classifier as ic
from src import intent_mapping


## 3.1 Load data

In [2]:
df = ic.load_hf_dataset()
print(df.shape)
print(df['intent'].value_counts())
df[['instruction', 'response', 'intent', 'category']].head()


(26872, 5)
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
delivery_options             995
delete_account               995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64


,instruction,response,intent,category
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,cancel_order,ORDER
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...,cancel_order,ORDER
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...,cancel_order,ORDER
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...,cancel_order,ORDER
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...,cancel_order,ORDER


## 3.2 Coarse routing taxonomy
The 27 fine intents collapse into 7 routing buckets (see `src/intent_mapping.py` for the full table).

In [3]:
for fine, coarse in list(intent_mapping.FINE_TO_COARSE.items())[:10]:
    print(f'{fine:>26s} -> {coarse}')
print('...')
print('\nRoute behavior per coarse bucket:', intent_mapping.ROUTE_BEHAVIOR)


                     greet -> greeting_goodbye_gratitude
                  greeting -> greeting_goodbye_gratitude
                       bye -> greeting_goodbye_gratitude
                   goodbye -> greeting_goodbye_gratitude
                     thank -> greeting_goodbye_gratitude
                 gratitude -> greeting_goodbye_gratitude
               track_order -> order_status
          delivery_options -> order_status
           delivery_period -> order_status
              cancel_order -> order_management
...

Route behavior per coarse bucket: {'greeting_goodbye_gratitude': 'direct', 'order_status': 'rag', 'order_management': 'rag', 'billing_and_refunds': 'rag', 'account_management': 'rag', 'complaint': 'escalate', 'out_of_scope': 'direct'}


## 3.3 Train fine-grained classifier
Word-level TF-IDF (1-2 grams, English stopwords removed) + `LinearSVC(class_weight='balanced')`.

In [4]:
pipeline, train_df, test_df = ic.train(df)


Fine-grained intent report:
                          precision    recall  f1-score   support

            cancel_order       0.99      0.98      0.99       200
            change_order       0.95      0.98      0.97       199
 change_shipping_address       0.99      1.00      0.99       195
  check_cancellation_fee       1.00      1.00      1.00       190
           check_invoice       0.82      0.87      0.84       200
   check_payment_methods       1.00      1.00      1.00       200
     check_refund_policy       1.00      0.98      0.99       199
               complaint       1.00      1.00      1.00       200
contact_customer_service       1.00      0.98      0.99       200
     contact_human_agent       0.99      0.99      0.99       200
          create_account       0.98      0.98      0.98       199
          delete_account       0.97      1.00      0.98       199
        delivery_options       0.99      1.00      1.00       199
         delivery_period       1.00      0.99  

## 3.4 Inference + coarse mapping + confidence

In [5]:
for msg in [
    "I want to cancel my order please",
    "This product is awful, I'm furious",
    "hi there!",
    "what's the weather like today",
    "how do I reset my password",
]:
    print(msg, '->', ic.predict_intent(pipeline, msg))


I want to cancel my order please -> {'fine_intent': 'cancel_order', 'coarse_intent': 'order_management', 'confidence': 0.26857344497717867}
This product is awful, I'm furious -> {'fine_intent': 'place_order', 'coarse_intent': 'order_management', 'confidence': 0.09021996245727924}
hi there! -> {'fine_intent': 'contact_human_agent', 'coarse_intent': 'out_of_scope', 'confidence': 0.04147923079446143}
what's the weather like today -> {'fine_intent': 'get_invoice', 'coarse_intent': 'billing_and_refunds', 'confidence': 0.050572247361171954}
how do I reset my password -> {'fine_intent': 'recover_password', 'coarse_intent': 'account_management', 'confidence': 0.36701198593653345}


## 3.5 Save model

In [6]:
os.makedirs('../models', exist_ok=True)
ic.save(pipeline, '../models/intent_classifier.joblib')
print('Saved.')


Saved.


## Notes / decisions to defend at assessment
- Trained on the gold `intent` column directly (recommended path in the brief) rather than
  zero/few-shot LLM prompting -- simpler, faster, fully reproducible, and the dataset already
  supports it.
- Word n-grams (not char n-grams) because intent lives in content words/phrases.
- `class_weight='balanced'` because the 27 intents are not evenly represented in real support
  logs (far more `track_order` than `delete_account`, for example).
- Fine-grained training + post-hoc coarse mapping keeps the fine signal available for future
  routing refinements, rather than baking the collapse into training directly.
- `complaint`/`review` are hard-routed to escalation-priority regardless of confidence -- this
  is a policy decision (safety-first for negative-experience messages), not a modeling
  limitation.